# [수학 Retreat #7] 자료의 정리와 대푯값: 혼돈 속에서 질서 세우기

이 주피터 노트북은 중학교 1학년 과정의 **자료의 정리와 해석** 단원을 파이썬 시각화 도구를 통해 다각도로 탐구하는 실습 교재입니다.
매일 쏟아지는 수많은 날것의 정보(Raw Data)로부터 어떻게 유의미한 패턴을 찾는지 학습합니다.

### 🟢 본 실습에서 다룰 두 가지 주제:
1. **자료의 정리 (`raw_data_analyzer`)**: 무작위 고객 행동 데이터나 매출 수치를 실시간 히스토그램으로 그리며, **계급의 크기(Bin Width)** 변화에 따라 숨겨진 패턴이 어떻게 보이는지 확인합니다.
2. **대푯값의 특성 (`representative_values`)**: 극단값(아웃라이어, Outlier)의 유무와 변동에 따라 **평균(Mean)**과 **중앙값(Median)**이 각각 어떻게 반응하고 중심을 지키는지 동적으로 비교합니다.

**⚠️ 안내:** 본 파일은 파이썬 초보자도 코드 한 줄 한 줄의 기하학적·수학적 의미를 이해할 수 있도록 **세포 단위의 멀티셀**로 쪼개어 구성했으며 상세한 한글 주석을 부착했습니다.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 데이터 처리를 위한 `numpy`, 3D/2D 인터랙티브 그래픽을 위한 `plotly`, 그리고 주피터 위젯 제어반 생성을 위한 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 라이브러리를 먼저 준비합니다.


In [ ]:
# !pip install 명령어는 주피터 노트북 안에서 터미널의 라이브러리 패키지 매니저를 구동하는 매직 커맨드입니다.
# -q (quiet) 옵션을 붙여 설치 상세 메시지를 생략하고 신속하게 진행합니다.
%pip install -q numpy plotly ipywidgets


### 1.2 모듈 로드 (Import)
기하 연산과 2차원/1차원 플로팅 및 위젯 구현을 위해 필요한 파이썬 모듈들을 임포트합니다.


In [ ]:
# numpy: 수치 데이터의 가공, 행렬 가공, 수학적 기술 통계량 연산의 기초가 되는 고성능 배열 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 마우스 호버 및 동적 갱신 시 매끄러운 렌더링을 보장하는 그래프 빌더 모듈입니다.
import plotly.graph_objects as go

# ipywidgets: 슬라이더 바, 드롭다운 상자 등 GUI 제어 위젯을 만들기 위한 라이브러리입니다.
import ipywidgets as widgets

# IPython.display: 테이블 형태의 예쁜 대시보드 리포트를 HTML로 노트북 내에 렌더링하기 위해 사용합니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 날것의 데이터 요약 및 도수분포표 분석 (`raw_data_analyzer`)
아무렇게나 흩어진 날것의 데이터(Raw Data)는 그 자체로 정보를 주지 못합니다.
이를 구간 단위인 **'계급(Class)'**으로 나누고 각 계급에 속하는 자료의 수인 **'도수(Frequency)'**를 센 표가 **도수분포표**이며, 이를 막대그래프로 나타낸 것이 **히스토그램**입니다.

여기서 **계급의 크기(Bin Width)**를 너무 작게 잡으면 노이즈가 많아 흐름을 읽기 어렵고, 너무 크게 잡으면 데이터 내부의 세밀한 특징들이 뭉개져 버립니다.
먼저, 250명의 가상 고객 결제 데이터(Lognormal Distribution 형태의 비대칭 분포)를 난수로 제너레이트하겠습니다.


In [ ]:
# 난수 생성의 일관성을 확보하기 위해 랜덤 시드(Seed)를 42로 고정합니다. (언제 실행해도 똑같은 난수 세트 보장)
np.random.seed(42)

# 현실세계의 매출 데이터와 같이 오른쪽으로 긴 꼬리를 갖는 비대칭 로그정규분포(Lognormal Distribution) 난수를 250개 생성합니다.
# mean: 로그 값 기준의 평균, sigma: 표준편차, size: 데이터 개수
raw_revenue_data = np.random.lognormal(mean=2.4, sigma=0.55, size=250)

# 가독성을 높이기 위해 만원 단위로 올림 및 반올림을 적용하여 정수형으로 변환합니다.
# 예: 12.34 -> 12(만원)
customer_spending = np.round(raw_revenue_data * 10).astype(int)

# 생성된 데이터의 맛보기 10개 및 기본 속성 출력
print(f"총 데이터 수: {len(customer_spending)}명")
print(f"최소 소비 금액: {np.min(customer_spending)}만 원 | 최대 소비 금액: {np.max(customer_spending)}만 원")
print("최초 10명의 날것의 데이터:", customer_spending[:10])


### 2.2 계급 크기에 따른 실시간 히스토그램 및 도수분포 대시보드 설계
사용자가 지정한 **계급의 크기 (Bin Width)**를 바탕으로 데이터 구간을 나누고, 도수분포표를 텍스트 대시보드로 요약하며, Plotly 히스토그램을 그리는 엔진 함수를 정의합니다.
이 히스토그램의 색상에는 브랜드 컬러의 청록색(`rgba(19, 168, 158, 0.85)`)을 적용합니다.


In [ ]:
def render_spending_histogram(bin_width):
    """
    선택된 계급의 크기(bin_width)를 바탕으로 데이터 분포를 도수분포표와 히스토그램으로 렌더링합니다.
    """
    # 1. 최솟값과 최댓값을 구하여 계급의 시작 범위와 끝 범위를 정의합니다.
    min_val = np.min(customer_spending)
    max_val = np.max(customer_spending)
    
    # 2. 계급의 크기(간격)에 맞춰 데이터 경계 범위를 생성합니다.
    # min_val부터 max_val + bin_width 범위까지 bin_width 크기만큼 건너뛰며 정렬합니다.
    bins = np.arange(0, max_val + bin_width + 1, bin_width)
    
    # 3. numpy.histogram 함수를 사용해 자동으로 도수분포표를 연산합니다.
    # counts: 각 계급 구간에 속하는 데이터의 개수 (도수)
    # edges: 계급의 양끝 경계값 목록
    counts, edges = np.histogram(customer_spending, bins=bins)
    
    # 4. 도수분포표 테이블 HTML 생성 (Premium UI 디자인 테마)
    table_rows_html = ""
    for i in range(len(counts)):
        # 도수가 0인 구간도 생략 없이 다 표시합니다.
        table_rows_html += f"""
        <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
            <td style='padding: 6px 12px;'>{edges[i]}만 원 이상 ~ {edges[i+1]}만 원 미만</td>
            <td style='padding: 6px 12px; text-align: center; font-weight: bold; color: #0B79D0;'>{counts[i]} 명</td>
            <td style='padding: 6px 12px; text-align: center; color: #666;'>{counts[i]/len(customer_spending)*100:.1f} %</td>
        </tr>
        """
        
    dashboard_html = f"""
    <div style='background: linear-gradient(135deg, rgba(11,121,208,0.06), rgba(47,168,93,0.06));
                padding: 20px; border-radius: 12px; border: 1.5px solid rgba(11,121,208,0.25);
                font-family: sans-serif; max-width: 500px; margin-bottom: 15px; box-shadow: 0 4px 12px rgba(0,0,0,0.04);'>
        <h4 style='margin: 0 0 10px 0; color: #0B79D0;'>📝 실시간 도수분포표 (계급의 크기: {bin_width}만 원)</h4>
        <table style='width: 100%; border-collapse: collapse; font-size: 0.9em;'>
            <thead>
                <tr style='background: rgba(11,121,208,0.1); border-bottom: 2px solid rgba(11,121,208,0.2);'>
                    <th style='padding: 8px 12px; text-align: left;'>계급 구간 (spending)</th>
                    <th style='padding: 8px 12px; text-align: center;'>도수 (Frequency)</th>
                    <th style='padding: 8px 12px; text-align: center;'>상대도수 (비율)</th>
                </tr>
            </thead>
            <tbody>
                {table_rows_html}
            </tbody>
        </table>
    </div>
    """
    display(HTML(dashboard_html))
    
    # 5. Plotly 히스토그램 트레이스 구성
    histogram_trace = go.Histogram(
        x=customer_spending,
        xbins=dict(
            start=0,      # 첫 번째 계급 시작점
            end=max_val + bin_width, # 마지막 계급 끝점
            size=bin_width # 계급의 크기 (폭)
        ),
        marker=dict(
            color='#13a89e',       # 브랜드 Cyan/Teal 컬러 적용
            line=dict(color='white', width=1.5) # 계급 막대 간 구분을 위한 흰색 테두리선
        ),
        opacity=0.85,
        hovertemplate='<b>계급 구간:</b> %{x}만 원대<br><b>도수(명):</b> %{y}명<extra></extra>', # 툴팁 커스터마이징
        name='고객 수'
    )
    
    # 6. 차트 레이아웃 스타일링
    layout = go.Layout(
        title=dict(
            text=f'<b>고객spending 분포 히스토그램 (계급의 크기: {bin_width}만 원)</b>',
            x=0.5,
            font=dict(size=16, color='#1F2937')
        ),
        xaxis=dict(title='고객 결제 금액 (만 원)', gridcolor='#E2E8F0', dtick=bin_width),
        yaxis=dict(title='도수 (고객 수, 명)', gridcolor='#E2E8F0'),
        plot_bgcolor='white',
        width=750, height=450,
        margin=dict(l=50, r=30, b=50, t=60)
    )
    
    fig = go.Figure(data=[histogram_trace], layout=layout)
    fig.show()


### 2.3 히스토그램 계급 제어 슬라이더 구동
계급의 크기(만원 단위)를 제어하는 정수 슬라이더 위젯을 생성하고 렌더러 함수에 연동하여 실행합니다.
슬라이더를 좌우로 움직이며 데이터 분포 요약 형태가 어떻게 세밀해지거나 뭉뚱그려지는지 사색해 보세요.


In [ ]:
# 계급의 크기를 조절하는 정수 슬라이더 위젯 생성 (1만원부터 15만원까지 1만원 간격 제어)
bin_width_slider = widgets.IntSlider(
    value=3, # 초기 설정값: 3만 원
    min=1, max=15, step=1,
    description='계급의 크기(만 원):',
    style={'description_width': 'initial'},
    continuous_update=False # 슬라이더 조작을 끝내고 마우스 손을 뗐을 때 그래프를 다시 그려 연산 부하를 최소화합니다.
)

# interact를 활용한 위젯 바인딩 구동
widgets.interact(render_spending_histogram, bin_width=bin_width_slider);


---
## 3. 실습 2: 대푯값의 특성 비교 시뮬레이터 (평균 vs 중앙값)
자료 전체의 특징을 대표적으로 나타내는 수치를 **대푯값**이라고 하며, 가장 널리 쓰이는 것이 **평균(Mean)**과 **중앙값(Median)**입니다.

- **평균(Mean)**: 모든 자료의 값을 더해 개수로 나눈 값으로, 데이터의 미세한 변화를 잘 반영하지만 **극단적인 아웃라이어(Outlier)**에 극도로 취약합니다.
- **중앙값(Median)**: 자료를 크기순으로 배열했을 때 정확히 한가운데(50% 위치)에 있는 값으로, 극단적인 값의 유무에 영향받지 않는 매우 **견고한(Robust) 성질**을 보입니다.

이번 실습에서는 1차원 수직선 상에 안정적인 데이터 분포를 그린 뒤, 마우스 슬라이더로 **극단적인 값(Outlier)**을 추가해 그 수치를 10부터 120까지 극단적으로 이동시킬 때 두 대푯값이 어떻게 요동치는지 눈으로 직접 비교합니다.


### 3.1 1차원 데이터 분포 및 실시간 대푯값 연산
기본적으로 분포된 9개의 데이터 세트 `[12, 14, 13, 15, 16, 14, 15, 17, 14]`에 슬라이더로 입력된 10번째 **극단값(Outlier)**을 병합한 뒤,
실시간으로 평균과 중앙값을 연산하여 수직선 상의 기하학적 선으로 매핑하는 함수를 구현합니다.


In [ ]:
# 9명의 기본 결제 데이터 정의 (평균 14~15대 수준의 안정한 군집 데이터)
baseline_spending = np.array([12, 14, 13, 15, 16, 14, 15, 17, 14])

def calculate_representative_points(outlier_value):
    """
    기본 데이터에 사용자가 슬라이더로 주입한 극단값을 붙여
    전체 데이터 세트의 평균(Mean)과 중앙값(Median)을 실시간으로 계산하여 반환합니다.
    """
    # 1. np.append 함수를 이용해 기본 배열의 끝에 극단값(outlier_value)을 결합합니다.
    combined_data = np.append(baseline_spending, outlier_value)
    
    # 2. 산술평균 계산
    mean_val = np.mean(combined_data)
    
    # 3. 중앙값 계산 (데이터를 정렬한 후 한가운데에 위치한 값 추출)
    median_val = np.median(combined_data)
    
    return combined_data, mean_val, median_val


### 3.2 수직선 1D 점 플로팅 및 대푯값 수직 지시선 플롯 구축
Plotly의 1차원 분산도(Scatter)를 정의하고, **평균의 위치는 빨간색 점선**, **중앙값의 위치는 녹색 실선**으로 수직을 세워 두 대표값의 이동 현상을 동적으로 시각화합니다.


In [ ]:
def render_representative_simulator(outlier_value):
    """
    아웃라이어 값이 주어졌을 때 전체 1D 데이터 점들과 평균선, 중앙값선을 캔버스에 그립니다.
    """
    # 1. 10번째 아웃라이어가 추가된 전체 데이터 및 통계량 계산
    data, mean_val, median_val = calculate_representative_points(outlier_value)
    
    # 2. 통계 지표 정보 출력용 대시보드 생성
    info_html = f"""
    <div style='background: white; border: 1.5px solid rgba(11, 121, 208, 0.2);
                padding: 16px; border-radius: 12px; max-width: 600px; margin-bottom: 10px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <div style='display: flex; justify-content: space-around; text-align: center;'>
            <div>
                <span style='color: #666; font-size: 0.9em;'>입력된 극단값(Outlier)</span>
                <h3 style='margin: 5px 0 0 0; color: #1F2937;'>{outlier_value}만 원</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #E11D48; font-size: 0.9em;'>🔴 산술 평균 (Mean)</span>
                <h3 style='margin: 5px 0 0 0; color: #E11D48;'>{mean_val:.2f}만 원</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #2FA85D; font-size: 0.9em;'>🟢 중앙값 (Median)</span>
                <h3 style='margin: 5px 0 0 0; color: #2FA85D;'>{median_val:.1f}만 원</h3>
            </div>
        </div>
    </div>
    """
    display(HTML(info_html))
    
    # 3. 1차원 데이터 시각화 점 구성 (y축은 모두 0으로 고정하여 1차원 수직선 상에 나열)
    # 기본 데이터 9개는 파란색 동그라미로 표기
    trace_base_points = go.Scatter(
        x=baseline_spending,
        y=np.zeros_like(baseline_spending),
        mode='markers',
        marker=dict(size=12, color='#0B79D0', opacity=0.7),
        name='기본 데이터 (9명)',
        hoverinfo='x'
    )
    
    # 조절되는 10번째 아웃라이어는 별도로 빨간색 큰 마커로 표기하여 구분합니다.
    trace_outlier_point = go.Scatter(
        x=[outlier_value],
        y=[0],
        mode='markers',
        marker=dict(size=16, color='#E11D48', symbol='star', line=dict(color='white', width=1.5)),
        name='극단값 (Outlier)',
        hoverinfo='x'
    )
    
    # 4. 평균의 위치를 관통하는 빨간 수직 점선(y = -0.5 to 0.5 범위) 정의
    trace_mean_line = go.Scatter(
        x=[mean_val, mean_val],
        y=[-0.5, 0.5],
        mode='lines',
        line=dict(color='#E11D48', width=3.5, dash='dash'),
        name='평균선 (Mean Line)'
    )
    
    # 5. 중앙값의 위치를 관통하는 녹색 수직 실선 정의
    trace_median_line = go.Scatter(
        x=[median_val, median_val],
        y=[-0.5, 0.5],
        mode='lines',
        line=dict(color='#2FA85D', width=3.5),
        name='중앙값선 (Median Line)'
    )
    
    # 6. 차트 레이아웃 스타일 설정 (1차원 수직선 형태를 강조하기 위해 y축 눈금은 숨김)
    layout = go.Layout(
        title=dict(
            text='<b>극단값 변화에 따른 평균과 중앙값의 반응성 비교 시뮬레이션</b>',
            x=0.5,
            font=dict(size=15)
        ),
        xaxis=dict(title='소비 금액 (만 원)', range=[8, 125], gridcolor='#F1F5F9', dtick=10),
        yaxis=dict(range=[-0.6, 0.6], showticklabels=False, showgrid=False, zeroline=True, zerolinecolor='#94A3B8'),
        plot_bgcolor='white',
        width=800, height=350,
        margin=dict(l=30, r=30, b=40, t=60),
        showlegend=True,
        legend=dict(orientation='h', y=-0.25, x=0.5, xanchor='center') # 범례를 하단 가로 방향 정렬
    )
    
    fig = go.Figure(data=[trace_base_points, trace_outlier_point, trace_mean_line, trace_median_line], layout=layout)
    fig.show()


### 3.3 대푯값 시뮬레이터 인터랙티브 위젯 기동
마우스로 극단값을 10(정상 범위)에서 120(초극단 아웃라이어)까지 동적으로 이동시킬 수 있는 정수 슬라이더를 생성합니다.
슬라이더를 우측으로 빠르게 이동시키면서, **빨간색 평균 점선은 자석에 끌려가듯 속절없이 우측으로 이동**하는 반면, **녹색 중앙값 실선은 한가운데에서 단단하게 버티는** 모습을 실시간으로 비교 체험해 보세요.


In [ ]:
# 극단값(Outlier)의 수치를 제어하는 슬라이더 위젯 생성
outlier_slider = widgets.IntSlider(
    value=15, # 시작값: 기본 데이터 군집 내부값
    min=10, max=120, step=2,
    description='극단값 수치 (만 원):',
    style={'description_width': 'initial'},
    continuous_update=True # 마우스 드래그 동작 중에 1차원 수직선과 지시선들이 실시간으로 즉각 반응해 요동칩니다.
)

# interact를 활용한 위젯 바인딩 구동
widgets.interact(render_representative_simulator, outlier_value=outlier_slider);


---
## 4. 깊이 있는 데이터 사색을 위한 질문 (Joshua를 위한 열린 질문)

1. **계급 크기(Bin Width)의 철학**:
   계급의 크기를 1만 원 단위로 극단적으로 잘게 쪼갰을 때와, 15만 원 단위로 투박하게 뭉뚱그렸을 때 정보 전달적인 측면에서 각각 어떤 딜레마(Dilemma)가 발생하나요? 의사결정권자 관점에서 최적의 정보 요약 수준은 어떻게 합리적으로 정할 수 있을지 사색해 보세요.

2. **평균주의 시스템 vs 중앙값적 중심**:
   극단적인 단 하나의 변수(Outlier)에 요동치며 중심축이 완전히 휩쓸려 나가는 '평균적 시스템'과, 극단적인 외부 충격과 노이즈 속에서도 자신의 굳건한 중심을 수호해 내는 '중앙값적 신념' 중, Joshua님의 현재 비즈니스 의사결정 및 기업 운영 시스템은 어느 쪽에 더 가깝게 설계되어 있나요? 기하학적 시뮬레이션의 움직임을 보며 느낀 철학적 소회를 나누어 주세요.
